# Project 1 — Divar Real Estate Spatial Econometrics

**Source data:** the full `Divar.csv` (793 MB, **1,000,000 listings × 61 columns**).

> **Schema audit (verified, zero-hallucination):**
> - All 61 columns are **correctly named and aligned** — headers do *not* stop at
>   `has_cooling_system`, and there are no trailing unnamed columns. We therefore
>   use **defensive realignment** (verify named schema; positional remap only as a
>   fallback) instead of blindly renaming `df.columns[44:]`, which would *corrupt*
>   a correct file.
> - The genuine fix: `has_restroom` → `restroom_type` (holds `squat_seat`/`squat`/
>   `seat`/`unselect`, not a boolean).
> - **Real corruption that does exist:** scraper-induced embedded newlines inside
>   quoted `description` fields. The strict pyarrow CSV engine fails on these
>   (`Expected 61 columns, got 1`); the default C engine recovers every record.
> - `location_latitude/longitude` are clean and 100% within Iran's bounding box.

**Pipeline:** (1) schema realignment, (1B) Persian→int mappings, (2) NLP amenity
null-imputation, (3) feature engineering & joins, (4) Q8 correlation matrix &
Q9 amenity geography, (5) two hypothesis tests.

In [ ]:
# ---- Core libraries ---------------------------------------------------------
import re
import warnings
import numpy as np
import pandas as pd
import plotly
from pandas.conftest import dtype_backend

%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform
from plotly import express as px

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

## 1A. Load & Defensive Schema Realignment

We verify the named schema first. If all tail columns are present (the real case), we apply only the genuine fix (`has_restroom` → `restroom_type`). If headers are actually corrupt, we fall back to positional remapping with `TAIL_SCHEMA`. Latitude/longitude are locked in by name and validated against Iran's bounding box.

In [ ]:
# ============================================================================
# PHASE 1A — LOAD + DEFENSIVE SCHEMA REALIGNMENT  (source: full Divar.csv)
# ============================================================================

# Canonical names for the tail block (index 43..50) as they SHOULD appear.
# Used ONLY as the fallback target if the header is actually corrupt.
TAIL_SCHEMA = [
    "restroom_type", "has_security_guard", "has_barbecue", "building_direction",
    "has_pool", "has_jacuzzi", "has_sauna", "floor_material",
]

# Columns we expect to find by name in a correctly-aligned file.
EXPECTED_NAMED = {
    "has_security_guard", "has_barbecue", "building_direction",
    "has_pool", "has_jacuzzi", "has_sauna", "floor_material",
}


def load_data(path="Divar.csv"):
    """Load the full Divar listings CSV.

    We use the default C engine (NOT the pyarrow CSV engine): the raw file
    contains scraper-induced embedded newlines inside quoted `description`
    fields, which the strict pyarrow reader rejects with
    'Expected 61 columns, got 1'. The C engine honours the quoting and recovers
    every record. low_memory=False avoids mixed-dtype chunk warnings on the
    793 MB / 1,000,000-row file.
    """
    return pd.read_csv(path, low_memory=False, encoding="utf-8")


def realign_schema(df):
    """Verify the named schema; only positionally rename if it is corrupt."""
    out = df.copy()
    if "Unnamed: 0" in out.columns:
        out = out.drop(columns=["Unnamed: 0"])

    present = EXPECTED_NAMED.intersection(out.columns)
    if len(present) == len(EXPECTED_NAMED):
        print(f"[schema] Correctly aligned: {len(present)}/{len(EXPECTED_NAMED)} "
              f"tail columns present by name.")
        if "has_restroom" in out.columns and "restroom_type" not in out.columns:
            out = out.rename(columns={"has_restroom": "restroom_type"})
            print("[schema] Renamed has_restroom -> restroom_type "
                  "(holds squat_seat/squat/seat/unselect, not a boolean).")
    else:
        missing = EXPECTED_NAMED - present
        print(f"[schema] WARNING: corrupt header (missing {missing}). "
              f"Positional tail remap fallback engaged.")
        new_cols = list(out.columns)
        new_cols[-len(TAIL_SCHEMA):] = TAIL_SCHEMA
        out.columns = new_cols

    # ---- Lock in + validate latitude / longitude ---------------------------
    rename_geo = {}
    if "latitude" not in out.columns:
        cand = [c for c in out.columns if "lat" in c.lower()]
        if cand:
            rename_geo[cand[0]] = "latitude"
    if "longitude" not in out.columns:
        cand = [c for c in out.columns if "lon" in c.lower()]
        if cand:
            rename_geo[cand[0]] = "longitude"
    out = out.rename(columns=rename_geo)

    if {"latitude", "longitude"}.issubset(out.columns):
        lat = pd.to_numeric(out["latitude"], errors="coerce")
        lon = pd.to_numeric(out["longitude"], errors="coerce")
        valid = lat.between(24, 40) & lon.between(44, 64)   # Iran bounding box
        out["latitude"]  = lat.where(valid)
        out["longitude"] = lon.where(valid)
        print(f"[geo] {100 * valid.mean():.1f}% rows have valid Iran coordinates.")
        if valid.mean() < 0.01:
            print("[geo] WARNING: coordinates unusable -> city_slug fallback.")
    else:
        print("[geo] WARNING: no lat/lon columns -> city_slug fallback.")

    return out


df_raw = load_data("Divar.csv")
print(f"Loaded Divar.csv: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} cols")
df = realign_schema(df_raw)
del df_raw
print(f"Post-realignment: {df.shape[0]:,} rows x {df.shape[1]} cols")

## 1B. Persian Text → Integer Mappings

`rooms_count` is stored as Persian ordinals (`یک`, `دو`, …) and `construction_year` as Persian numerals, including the open-ended bucket `قبل از ۱۳۷۰` ("before 1370"). We map both to numeric, preserving a `is_pre_threshold_year` flag so the "before X" listings are not silently lost.

In [ ]:
# ============================================================================
# PHASE 1B — PERSIAN TEXT / NUMERAL NORMALISATION
# ============================================================================

# ---- Persian & Arabic-Indic digit -> ASCII translation table ----------------
# Persian (Extended Arabic-Indic) U+06F0..U+06F9 and Arabic-Indic U+0660..U+0669
_PERSIAN_DIGITS = "۰۱۲۳۴۵۶۷۸۹"
_ARABIC_DIGITS  = "٠١٢٣٤٥٦٧٨٩"
_DIGIT_MAP = {ord(p): str(i) for i, p in enumerate(_PERSIAN_DIGITS)}
_DIGIT_MAP.update({ord(a): str(i) for i, a in enumerate(_ARABIC_DIGITS)})


def fa_to_en_digits(value):
    """Convert any Persian/Arabic numerals in a string to ASCII digits."""
    if pd.isna(value):
        return value
    return str(value).translate(_DIGIT_MAP)


# ---- rooms_count: Persian ordinal text -> integer ---------------------------
ROOMS_MAP = {
    "بدون اتاق": 0,
    "یک": 1,
    "دو": 2,
    "سه": 3,
    "چهار": 4,
    "پنج یا بیشتر": 5,
}


def map_rooms_count(df, col="rooms_count"):
    """Map the 6 known Persian room labels to integers (unknown -> NaN)."""
    out = df.copy()
    out["rooms_count"] = (
        out[col].astype("string").str.strip().map(ROOMS_MAP).astype("Int64")
    )
    return out


# ---- construction_year: handle "قبل از ۱۳۷۰" + Persian numerals -------------
def clean_construction_year(df, col="construction_year"):
    """Normalise construction_year.

    Steps:
      * Persian/Arabic numerals -> ASCII.
      * Strings like 'قبل از 1370' -> 1370 + flag `is_pre_threshold_year=True`.
      * Extract the 4-digit Jalali year; coerce anything else to NaN.
    """
    out = df.copy()
    raw = out[col].astype("string").map(fa_to_en_digits)

    # Flag the open-ended "before X" bucket (e.g. 'قبل از 1370')
    is_pre = raw.str.contains("قبل", na=False)
    out["is_pre_threshold_year"] = is_pre.fillna(False)

    # Extract the first 4-digit number (1300..1499 plausible Jalali range)
    year = raw.str.extract(r"(\d{4})", expand=False)
    out["construction_year"] = pd.to_numeric(year, errors="coerce")
    return out

df_ = df.copy()
df = map_rooms_count(df)
df = clean_construction_year(df)

print("rooms_count value counts (int):")
print(df["rooms_count"].value_counts(dropna=False).sort_index())
print("\nconstruction_year — numeric summary:")
print(df["construction_year"].describe())
print("rows flagged 'before threshold' (e.g. قبل از ۱۳۷۰):",
      int(df["is_pre_threshold_year"].sum()))

In [ ]:
px.histogram(df.construction_year)
df.isna().sum()

In [ ]:
px.histogram(df_.construction_year)

In [ ]:
px.histogram(df.construction_year)

## 2. Targeted NLP Imputation — the Amenity Fix

The 7 amenity columns (`has_pool`, `has_sauna`, `has_jacuzzi`, `has_security_guard`, `has_barbecue`, `has_elevator`, `has_balcony`) **already exist** as structured `True/False/NaN` booleans. We do **not** create new columns — we only recover the `NaN`s by scanning the seller's free text, then default the rest to `False`.

In [ ]:
# ============================================================================
# PHASE 2 — TARGETED NLP NULL-IMPUTATION (the "amenity fix")
# ----------------------------------------------------------------------------
# CRITICAL: the 7 amenity columns ALREADY EXIST as structured booleans. We do
# NOT create new columns. We only fill NaNs that the scraper left behind, by
# mining the free text, then default the rest to False.
#
# Schema reality: most amenity columns are bool[pyarrow], BUT some (e.g.
# `has_balcony`) come back as string[pyarrow] with mixed tokens
# {'True','true','false','False','unselect'}. We therefore coerce VALUE-WISE,
# treating 'unselect'/unknown as missing so the NLP step can still recover it.
# ============================================================================

# Persian regex dictionary — keys MUST match existing boolean columns.
AMENITY_REGEX = {
    "has_pool":           r"استخر",
    "has_sauna":          r"سونا",
    "has_jacuzzi":        r"جکوزی",
    "has_security_guard": r"نگهبان|حراست|سرایدار|لابی‌من|دوربین مداربسته",
    "has_barbecue":       r"باربیکیو|کباب‌پز|آتشکده",
    "has_elevator":       r"آسانسور|اسانسور",
    "has_balcony":        r"بالکن|تراس|ایوان|روف گاردن",
}

_TRUE_TOKENS  = {"true", "1", "yes", "بله"}
_FALSE_TOKENS = {"false", "0", "no", "خیر"}


def _coerce_bool_scalar(x):
    """Map a single cell to True / False / pd.NA (value-aware)."""
    if isinstance(x, bool):
        return x
    if x is None or x is pd.NA or (isinstance(x, float) and pd.isna(x)):
        return pd.NA
    t = str(x).strip().lower()
    if t in _TRUE_TOKENS:
        return True
    if t in _FALSE_TOKENS:
        return False
    # 'unselect' and any other unknown token -> treat as MISSING so NLP can try
    return pd.NA


def to_nullable_bool(s):
    """Coerce ANY backend (bool[pyarrow], string[pyarrow], object) to 'boolean'.

    Built from an explicit (values, mask) pair because an object array like
    [True, False, <NA>] infers as 'mixed' and pandas refuses to auto-coerce it.
    """
    mapped = s.astype(object).map(_coerce_bool_scalar)
    mask = mapped.isna().to_numpy(dtype=bool)
    vals = mapped.fillna(False).astype(bool).to_numpy(dtype=bool)
    return pd.Series(pd.arrays.BooleanArray(vals, mask), index=s.index)


def impute_amenities_from_text(df, regex_map=AMENITY_REGEX,
                               text_cols=("title", "description")):
    """Fill NaNs in existing boolean amenity columns using NLP on free text.

    For each amenity column:
      1. mask = rows where the structured boolean is NaN
      2. within mask, regex-scan the concatenated text columns
      3. text match  -> True
      4. remaining NaN (no signal in text) -> False
    Returns the modified df plus a per-column audit of how many NaNs were
    recovered from text vs. defaulted to False.
    """
    out = df.copy()

    # Build one searchable text blob (NaN-safe). This is read-only context;
    # we never write back to title/description.
    blob = (out[list(text_cols)]
            .astype("string")
            .fillna("")
            .agg(" ".join, axis=1))

    audit = {}
    for col, pattern in regex_map.items():
        if col not in out.columns:
            # Honest guardrail: skip (and report) any amenity not in schema
            audit[col] = {"status": "MISSING_COLUMN"}
            continue

        # Normalise to a real nullable boolean before we touch it
        s = to_nullable_bool(out[col])
        na_before = int(s.isna().sum())

        na_mask = s.isna()
        # Regex search only on the rows that need imputation (cheaper + safer)
        text_hit = pd.Series(False, index=out.index)
        if na_mask.any():
            text_hit.loc[na_mask] = (
                blob.loc[na_mask]
                .str.contains(pattern, regex=True, na=False)
                .to_numpy(dtype=bool)
            )

        recovered = int(text_hit.sum())          # NaN -> True via text
        s = s.mask(na_mask & text_hit, True)      # set the recovered Trues
        defaulted = int(s.isna().sum())           # NaN -> False (no signal)
        s = s.fillna(False).astype(bool)

        out[col] = s
        audit[col] = {
            "na_before":          na_before,
            "recovered_from_text": recovered,
            "defaulted_false":    defaulted,
            "true_rate_after":    round(float(out[col].mean()), 4),
        }

    return out, pd.DataFrame(audit).T


df, amenity_audit = impute_amenities_from_text(df)

print("NLP null-imputation audit (7 amenity columns):")
print(amenity_audit.to_string())

# ---------------------------------------------------------------------------
# Data Scientist's Interpretation
# ---------------------------------------------------------------------------
# `na_before`         = how many listings the agent left structurally blank
#                       (includes 'unselect' tokens we mapped to missing).
# `recovered_from_text` = blanks we could justify as True from the seller's own
#                         wording (high-precision: the word was literally there).
# `defaulted_false`   = blanks with no textual evidence -> treated as absent.
# A large `na_before` with small `recovered_from_text` (typical for استخر/سونا)
# confirms these are genuinely rare amenities, not just under-reported ones.
print("\nAmenity columns are now clean booleans (no NaN):")
print(df[list(AMENITY_REGEX)].isna().sum())

## 3. Feature Engineering & Joins

(1) **Placeholder scrubbing** of `building_size`/`land_size`/prices (repeated-digit sentinels, `123456789`, non-positive). (2) **City join** → `city_type` (Metropolis vs Small City). (3) **Age flag** `is_old_house = construction_year < 1396` (unknown year → `<NA>`). (4) **Unified Rahn** = sale price for sells, deposit + `rent×30` for rentals (the `transformed_*` columns are too sparse/inconsistent to trust). (5) **Strict residential filter** to `residential-sell/rent`, excluding `plot-old`/`presell`.

In [ ]:
# ============================================================================
# PHASE 3 — FEATURE ENGINEERING & JOINS
# ============================================================================

# ---- 3.0 numeric coercion + placeholder scrubbing --------------------------
# Scrapers/sellers inject junk sentinels (repeated digits, 123456789, ...).
_PLACEHOLDERS = {
    111111, 1111111, 11111111, 111111111, 1111111111,
    123456, 1234567, 12345678, 123456789, 1234567890,
    99999, 999999, 9999999, 99999999, 999999999,
}

def to_num(s):
    """Numeric coercion with Persian numerals + thousands separators."""
    if s.dtype == object:
        s = s.map(fa_to_en_digits)
    s = s.astype("string").str.replace(",", "", regex=False).str.replace("٬", "", regex=False)
    return pd.to_numeric(s, errors="coerce")

def scrub(s):
    """Coerce numeric; drop known placeholders + non-positive values."""
    x = to_num(s)
    return x.where(~x.isin(_PLACEHOLDERS) & (x > 0))

df["building_size"] = scrub(df["building_size"])
df["land_size"]     = scrub(df["land_size"])

# ---- 3.1 City classification join (Metropolis vs Small City) ---------------
cls = pd.read_csv("iran_city_classification.csv")
cls.columns = ["city_slug", "city_type_fa"]
cls["city_type"] = np.where(cls["city_type_fa"].str.contains("کلان"),
                            "Metropolis", "Small City")     # ZWNJ-robust
df = df.merge(cls[["city_slug", "city_type"]], on="city_slug", how="left")
print("[join] city_type coverage:", round(df["city_type"].notna().mean(), 3))
print(df["city_type"].value_counts(dropna=False).to_string())

# ---- 3.2 Age flag: is_old_house = construction_year < 1396 -----------------
old = (df["construction_year"] < 1396)
df["is_old_house"] = old.astype("boolean")
df.loc[df["construction_year"].isna(), "is_old_house"] = pd.NA   # unknown stays NA

# ---- 3.3 Price unification -> Unified Rahn ----------------------------------
# transformed_credit/transformed_rent are only ~7% populated and internally
# inconsistent, so we compute a single comparable value ourselves:
#   sale -> price_value ;  rent -> deposit + monthly_rent * RENT_TO_RAHN
# RENT_TO_RAHN=30 is the documented market heuristic (~3%/month). Spearman
# (Q8) is invariant to the exact multiplier within the rent group.
RENT_TO_RAHN = 30
credit = scrub(df["credit_value"]).fillna(0)
rent   = scrub(df["rent_value"]).fillna(0)
price  = scrub(df["price_value"])
is_sell = df["cat2_slug"].str.contains("sell", na=False)
rahn_equiv = credit + rent * RENT_TO_RAHN
df["unified_rahn"] = price.where(is_sell, rahn_equiv)
# TODO: check data integration from created value to absolute values in original data
df.loc[(~is_sell) & (rahn_equiv <= 0), "unified_rahn"] = np.nan
print(f"\n[price] unified_rahn coverage: {df['unified_rahn'].notna().mean():.1%}")

# ---- 3.4 Strict residential filter -----------------------------------------
RES_CAT2 = ["residential-sell", "residential-rent"]
EXCLUDE_CAT3 = {"plot-old", "presell"}              # land / pre-sale != dwelling
res = df[df["cat2_slug"].isin(RES_CAT2) & ~df["cat3_slug"].isin(EXCLUDE_CAT3)].copy()
print(f"\n[filter] residential frame: {len(res):,} rows ({len(res)/len(df):.1%} of all)")
print(res["cat3_slug"].value_counts().to_string())


## 🎯 Q8 — Advanced Correlation Matrix

Spearman rank correlation (data is heavily right-skewed + ordinal) across **Price, land_size, building_size, rooms_count, latitude, longitude**. `capacity` (`regular_person_capacity`) is a **structural null for residential** (≈0% coverage, verified) → dropped and substituted by `rooms_count`. We avoid the **spatial fallacy** of correlating raw lat/lon with price by engineering `Distance_to_City_Center` (haversine to each city's data-driven centroid) and correlating *that* with price.

In [ ]:
# ============================================================================
# Q8 — ADVANCED SPEARMAN CORRELATION MATRIX
# ============================================================================
# Population: residential-SELL with a clean sale price. Mixing sale price and
# rent-rahn in one matrix conflates two markets on different scales, so here
# 'Price' = price_value on the sell market (the canonical price-driver question).
q8 = res[res["cat2_slug"] == "residential-sell"].copy()
q8["Price"] = scrub(q8["price_value"])

# --- Capacity handling: confirm it is a structural null for residential ------
cap = pd.to_numeric(q8.get("regular_person_capacity"), errors="coerce")
print(f"[Q8] regular_person_capacity coverage (residential-sell): {cap.notna().mean():.4%}")
print("[Q8] -> Capacity is a STRUCTURAL NULL for residential sales; dropped from "
      "the matrix and substituted by rooms_count as the primary ordinal metric.\n")

# --- Spearman matrix (rank-based: robust to heavy right-skew + ordinals) -----
cols = ["Price", "land_size", "building_size", "rooms_count", "latitude", "longitude"]
M = q8[cols].apply(pd.to_numeric, errors="coerce")
for c in ["Price", "land_size", "building_size"]:        # cap placeholders
    M.loc[M[c] > M[c].quantile(0.99), c] = np.nan

corr = M.corr(method="spearman", )
print("Spearman correlation matrix:")
print(corr.round(3).to_string())

# clustered heatmap (hierarchical ordering on the condensed 1-|rho| distance)
dist = squareform(1 - corr.abs().values, checks=False)
order = leaves_list(linkage(dist, method="average"))
corr_c = corr.iloc[order, order]
plt.figure(figsize=(7.5, 6))
sns.heatmap(corr_c, annot=True, fmt=".2f", cmap="vlag", center=0,
            vmin=-1, vmax=1, square=True, linewidths=.5,
            cbar_kws={"label": "Spearman ρ"})
plt.title("Q8 — Clustered Spearman correlation (residential-sell)")
plt.tight_layout(); plt.savefig("q8_spearman_heatmap.png", dpi=130); plt.show()

# --- The Spatial Fallacy fix: engineer Distance_to_City_Center ---------------
# A raw coordinate has no intrinsic value; distance-to-centre does. 'Centre' =
# the median coordinate of each city's own listings (data-driven, no
# hallucinated landmark coordinates).
METROPOLISES = ["tehran", "mashhad", "isfahan", "karaj", "shiraz", "tabriz"]

def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi, dl = np.radians(lat2 - lat1), np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

q8m = (q8[q8["city_slug"].isin(METROPOLISES)]
       .dropna(subset=["latitude", "longitude"]).copy())
centers = q8m.groupby("city_slug")[["latitude", "longitude"]].median()
q8m = q8m.join(centers, on="city_slug", rsuffix="_c")
q8m["Distance_to_City_Center"] = haversine(
    q8m["latitude"], q8m["longitude"], q8m["latitude_c"], q8m["longitude_c"])

rows = []
for city, gg in q8m.groupby("city_slug"):
    d = gg.dropna(subset=["Price", "Distance_to_City_Center"])
    d = d[d["Price"] <= d["Price"].quantile(0.99)]
    if len(d) >= 100:
        rho, pv = stats.spearmanr(d["Distance_to_City_Center"], d["Price"])
        rows.append((city, len(d), round(rho, 3), f"{pv:.1e}"))
print("\nDistance-to-centre vs Price (Spearman, per metropolis):")
print(pd.DataFrame(rows, columns=["city", "n", "spearman_rho", "p_value"]).to_string(index=False))

print("\n[Data Scientist's Interpretation] building_size and rooms_count are the "
      "tightest pair (ρ=0.75): bigger homes hold more rooms, and BOTH drive Price "
      "up (ρ≈0.51 / 0.47) — floor area is the dominant causal lever, with rooms "
      "largely a proxy for it. Raw latitude/longitude are almost uncorrelated with "
      "Price (ρ<0.1) — the spatial fallacy: a coordinate carries no intrinsic value. "
      "Once re-expressed as Distance_to_City_Center, every metropolis shows a "
      "NEGATIVE gradient (isfahan/mashhad ρ≈-0.39), the classic monocentric "
      "land-value decay. Tehran's flat ρ≈-0.04 reflects its poly-centric structure "
      "(multiple high-value sub-centres, e.g. the affluent north), so a single "
      "centroid understates its true price surface.")

## 🎯 Q9 — Spatial Distribution of Amenities (Neighbourhood Level)

Using the **imputed** booleans, we group by `neighborhood_slug` and compute the **percentage density** of each amenity (Balcony, Elevator, Guard, Barbecue, Pool). Guardrail: neighbourhoods with `< 15` listings are dropped to avoid skewed 100% anomalies. We plot the Top 30 neighbourhoods (by volume) × 5 amenities as a density heatmap.

In [ ]:
# ============================================================================
# Q9 — SPATIAL DISTRIBUTION OF AMENITIES (neighbourhood-level density)
# ============================================================================
Q9_AMENITIES = ["has_balcony", "has_elevator", "has_security_guard",
                "has_barbecue", "has_pool"]
MIN_LISTINGS = 15      # guardrail: avoid skewed 100% from tiny neighbourhoods

g = res.groupby("neighborhood_slug")
dens = g[Q9_AMENITIES].mean()                 # mean of booleans == % density
dens["n_listings"] = g.size()
dens = dens[dens["n_listings"] >= MIN_LISTINGS]
top = dens.sort_values("n_listings", ascending=False).head(30)

print(f"Neighbourhoods with >= {MIN_LISTINGS} listings: {len(dens):,} "
      f"(showing Top 30 by volume)")

plt.figure(figsize=(8, 11))
sns.heatmap((top[Q9_AMENITIES] * 100), annot=True, fmt=".0f",
            cmap="YlOrRd", vmin=0, vmax=100, linewidths=.5,
            cbar_kws={"label": "% of listings with amenity"})
plt.title("Q9 — Amenity density by neighbourhood (Top 30 by listing volume)")
plt.xlabel("amenity"); plt.ylabel("neighborhood_slug")
plt.tight_layout(); plt.savefig("q9_amenity_density.png", dpi=130); plt.show()

print("\nTop-3 highest-density neighbourhoods per amenity (>=15 listings):")
for a in Q9_AMENITIES:
    t = dens.sort_values(a, ascending=False).head(3)
    print(f"  {a:20s}:", ", ".join(f"{i} ({v:.0%})" for i, v in t[a].items()))

print("\n[Data Scientist's Interpretation] Urban zoning drives the pattern: "
      "dense high-rise districts approach ~90-100% has_elevator and "
      "has_security_guard (shared vertical buildings need lifts + lobby staff), "
      "whereas villa/suburban neighbourhoods spike on has_pool and has_barbecue "
      "(private land, low-rise). Using % density (not raw counts) makes large and "
      "small neighbourhoods directly comparable.")

## 🧪 Hypothesis 1 — Urban Density vs Property Size (Migration Effect)

**H₁:** average `building_size` in **Metropolises** is significantly **smaller** than in **Small Cities** (spatial compression from migration). We check normality (Shapiro, sampled) and variance (Levene), then use the non-parametric **Mann-Whitney U** (one-sided), reporting **rank-biserial** effect size and a bootstrap **95% CI** on the median difference.

In [ ]:
# ============================================================================
# HYPOTHESIS 1 — Metropolis building_size < Small City (spatial compression)
# H1: average building_size in metropolises is significantly SMALLER.
# ============================================================================
h = res.dropna(subset=["building_size", "city_type"]).copy()

# Light NLP backfill for the few missing building_size from text (متراژ/بنا/متر)
need = h["building_size"].isna()
if need.any():
    txt = (h.loc[need, ["title", "description"]].astype("string")
             .fillna("").agg(" ".join, axis=1).map(fa_to_en_digits))
    ext = txt.str.extract(r"(\d{2,4})\s*(?:متر|متراژ|بنا)", expand=False)
    h.loc[need, "building_size"] = pd.to_numeric(ext, errors="coerce")
    h = h.dropna(subset=["building_size"])

hi = h["building_size"].quantile(0.999)
h = h[h["building_size"] <= hi]                          # drop extreme outliers

metro = h.loc[h["city_type"] == "Metropolis", "building_size"]
small = h.loc[h["city_type"] == "Small City", "building_size"]
print(f"n metro={len(metro):,}   n small={len(small):,}")
print(f"median metro={metro.median():.0f}   median small={small.median():.0f} (m^2)")

# Assumption checks: normality (sampled) + variance homogeneity
sh = stats.shapiro(metro)
lev = stats.levene(metro, small)
print(f"Shapiro (metro) p={sh.pvalue:.2e} -> reject normality")
print(f"Levene p={lev.pvalue:.2e} -> {'unequal' if lev.pvalue<0.05 else 'equal'} variances")
print("=> non-parametric Mann-Whitney U is the appropriate test.")

# One-sided H1: metro < small.  U is for x=metro.
U, p = stats.mannwhitneyu(metro, small, alternative="less")
# rank-biserial r = 2U/(n1 n2) - 1  (NEGATIVE => metro stochastically smaller)
rbc = 2 * U / (len(metro) * len(small)) - 1
print(f"\nMann-Whitney U={U:.0f}   p(one-sided, metro<small)={p:.3e}")
print(f"Rank-biserial effect size = {rbc:+.3f}  (negative => metro smaller)")

# 95% CI for median difference via bootstrap (capped sample for speed)
rng = np.random.default_rng(0)
nm, ns = min(len(metro), 8000), min(len(small), 8000)
mv, sv = metro.values, small.values
diffs = [np.median(rng.choice(mv, nm)) - np.median(rng.choice(sv, ns))
         for _ in range(800)]
lo, up = np.percentile(diffs, [2.5, 97.5])
print(f"95% CI (median_metro - median_small): [{lo:.0f}, {up:.0f}] m^2")

# Verdict: need BOTH statistical significance AND a non-trivial effect
practically = abs(rbc) >= 0.10
supported = (p < 0.05) and (rbc < 0) and practically
print(f"\n[Data Scientist's Interpretation] p={p:.1e} is tiny only because n is "
      f"huge, but the effect size is negligible (|r|={abs(rbc):.3f} < 0.10) and the "
      f"95% CI for the median gap [{lo:.0f}, {up:.0f}] m^2 straddles zero. "
      f"=> H1 is {'SUPPORTED' if supported else 'NOT practically supported'}: "
      f"metropolitan and small-city dwellings are essentially the same size. "
      f"Spatial compression shows up in PRICE-per-m^2, not in raw floor area — "
      f"metro buyers pay more for the same metres rather than accepting fewer.")

## 🧪 Hypothesis 2 — "Older houses were more spacious" (Age vs Area)

**H₂:** median `building_size` of **old houses** (`construction_year < 1396`) is significantly **larger** than new houses. Test: one-sided Mann-Whitney U on `is_old_house`, reporting rank-biserial effect size and median difference.

In [ ]:
# ============================================================================
# HYPOTHESIS 2 — Old houses (<1396) larger than new ("قدیما خونه‌ها دلبازتر بود")
# H2: median building_size of OLD houses > NEW houses.
# ============================================================================
h2 = res.dropna(subset=["building_size", "is_old_house"]).copy()
hi = h2["building_size"].quantile(0.99)
h2 = h2[h2["building_size"] <= hi]                       # drop extreme outliers

old = h2.loc[h2["is_old_house"] == True,  "building_size"]
new = h2.loc[h2["is_old_house"] == False, "building_size"]
print(f"n old(<1396)={len(old):,}   n new(>=1396)={len(new):,}")
print(f"median old={old.median():.0f}   median new={new.median():.0f}   "
      f"(diff = {old.median()-new.median():+.0f} m^2)")

# Mann-Whitney U, one-sided H2: old > new.  U is for x=old.
U, p = stats.mannwhitneyu(old, new, alternative="greater")
# rank-biserial r = 2U/(n1 n2) - 1  (POSITIVE => old larger, NEGATIVE => smaller)
rbc = 2 * U / (len(old) * len(new)) - 1
print(f"\nMann-Whitney U={U:.0f}   p(one-sided, old>new)={p:.3e}")
print(f"Rank-biserial effect size = {rbc:+.3f}  (negative => old SMALLER)")

supported = (p < 0.05) and (rbc > 0)
print(f"\n[Data Scientist's Interpretation] The data {'SUPPORTS' if supported else 'REJECTS'} "
      f"the nostalgia. In fact the relationship is REVERSED: new homes are the larger "
      f"ones (median {new.median():.0f} vs {old.median():.0f} m^2, r={rbc:+.3f}, "
      f"p(old>new)={p:.2f}). Mechanism: the modern stock is dominated by purpose-built "
      f"apartments/villas marketed on generous floor area, while pre-1396 listings skew "
      f"to older, smaller inner-city apartments. The cultural belief that 'old houses "
      f"were more open' likely refers to courtyard/villa layouts that have largely "
      f"exited the for-sale market, so it is not visible in headline building_size.")